# 第 10 章 · 用 vLLM 提供服务

> 09 章看的是内部结构；这一章把服务真正跑起来：Mac 上用 `vllm-metal`，云上用官方镜像，然后用 `httpx` 把 OpenAI 兼容 API 逐个打一遍。

业务方在群里问：「模型能不能接进我们 App？」你还没想好怎么答——把一个 Python 类给他，
还是把一个 URL 给他？第一种，每来一个新业务方，你就要陪他调一遍 tokenize、流式拼接和
`usage` 计数；第二种，他照着 OpenAI 文档写，你只要把服务守住。

这一章就是把「类」变成「URL」：先起服务，再逐个接口验证契约，最后算清楚这台服务养不养得起。

| 项目 | 内容 |
| --- | --- |
| 预计用时 | 75–90 分钟 |
| 运行环境 | 🏠 Mac（有 `vllm-metal` 时在线）· 无服务时自动回放 `cloud/fixtures/` |
| 前置章节 | 09（vLLM 分层与请求生命周期） |
| 本章产物 | 部署拓扑、三套启动命令、OpenAI 接口契约表、10 条 smoke test、显存/成本估算 |
| 本章接口 | `VLLMClient.chat / embed / mode`；无服务时返回同构的回放结果 |

**学完你能回答：**
1. 在 Mac 和云上分别怎么起一个 OpenAI 兼容服务？关键参数有哪些？
2. `/v1/chat/completions`、`/v1/completions`、`/v1/embeddings` 的请求和响应字段分别是什么？
3. LoRA 多适配器、结构化输出、工具调用分别要开哪些参数？
4. Qwen3-8B / 27B 在 24/48/80GB 卡上、不同精度下能开多少并发？每百万 token 大概多少钱？

## 10.0 本章目标与路线

**一句话目标**：把 vLLM 服务真正跑起来（Mac 用 `vllm-metal`、云上用 Docker），
并用一个「有服务走 HTTP、没服务回放 fixture」的客户端逐个验证 OpenAI 兼容接口的契约。

**代价与边界**：
- 本地没有 vLLM 服务，所以本章默认走**回放**；所有在线代码路径都在，但没有服务时不会执行；
- 不 pip 安装、不联网、不下载模型；回放数据来自 `cloud/fixtures/`（示例录制，非本机实测）；
- 显存/成本表是**公式估算**，用示例价格与回放吞吐，绝对值以真机为准。

**本章交付物**：

| # | 交付物 | 在哪一节 | 怎么验收 |
| --- | --- | --- | --- |
| 1 | Mac / 云上 / compose 三套启动命令 | 10.3 Step 1 | 命令能打印，关键参数齐全 |
| 2 | 可降级的 `VLLMClient`（httpx + fixture） | 10.3 Step 2–3 | 无服务时返回 `replay=True` 且不报错 |
| 3 | 10 条 smoke test | 10.4 实验 A | 9 条 200 + 1 条 400，字段完整 |
| 4 | 显存账本 + 成本估算 | 10.4 实验 B/C | 8B/27B × 精度 × 卡型可复算 |
| 5 | 接口契约表 + vLLM 实现映射 | 10.2 / 10.5 | 每个 endpoint 能说出请求/响应字段 |

**阅读路线**：10.1 问题 → 10.2 高阶设计（拓扑 + 契约）→ 10.3 分四步实现
→ 10.4 实验 A/B/C → 10.5 对照 vLLM → 10.6 练习。

## 10.1 问题与直觉：手工调用 vs HTTP 服务

09 章你已经看到了内部对象。如果业务方直接用 Python 调 engine，会发生什么？
每个调用方都要自己处理 tokenize、流式拼接、`usage` 计数、重试、错误码、并发限流；
换模型时全部客户端重写——这是「没有协议」的代价。

vLLM 的做法是在引擎外面加一层 OpenAI 兼容 API Server，把内部对象翻译成标准 JSON：
`messages` 进、`choices[].message` 出、流式走 SSE、错误码是 400/422/429。

这一层最贴切的比喻是**海关**：外面的世界说 OpenAI 方言，里面的引擎说 `Request` /
`SchedulerOutput` 方言；翻译、报关、退单全在海关发生，引擎不需要知道外面是谁。
反过来，海关出了错，外面收到的就是一张看不懂的罚单——**所以错误码必须在这一层翻译干净**。

部署拓扑与 API 契约放在 10.2 一起看。先记住一个反例：不接兼容层，每个业务方都要自己
实现 tokenize、流式拼接、重试和 token 计数；换模型时全部客户端重写——**协议的价值
就是把变化关在服务端**。

| 维度 | 手工调用引擎 | HTTP + OpenAI 兼容 API |
| --- | --- | --- |
| 调用方式 | Python 对象、进程内 | HTTP/JSON，跨语言跨进程 |
| 流式 | 自己实现生成器 | SSE，标准 `data:` 帧 |
| 错误处理 | 抛异常 | 状态码 400/422/429/503 |
| 多租户 | 自己做鉴权限流 | `--api-key` + 网关 |
| 换模型 | 改调用方代码 | 改服务端 `--model` |

「换模型只改服务端 `--model`」这一行，是整张表里最省钱的一行：没有它，每换一次模型，
有多少个调用方就有多少个 PR 要发。

本章要解决的两个子问题：

- **契约正确性**：三种 endpoint 的请求/响应字段、状态码、流式格式是否与 OpenAI 一致？
- **成本**：给定模型 / 精度 / 卡型，能开多少并发？每百万 token 多少钱？

运行环境的选择也影响契约验证方式：有服务就测真实 HTTP，没服务就回放录制响应——
但**断言完全相同**，这是本章能在 Mac 上跑通的关键。

## 10.2 高阶设计：部署拓扑、API 契约与客户端降级

先看部署拓扑：

![](../assets/figures/ch10_topology.svg)

图里三个要点：

1. **两种部署形态共用同一套 API**：Mac 单进程 Metal worker，云上「API+EngineCore 容器 + 多 GPU worker」，客户端代码不变；
2. **能力是叠加的**：LoRA 适配器、embeddings 实例、Prometheus 指标都是并列附加件，不是重写服务；
3. **关机是流程的一部分**：按小时计费的卡忘关一晚，预算就没了。

看图时多留意两样东西：箭头上的协议（HTTP/SSE，还是进程内 RPC），以及哪些框是**可选的**——
embeddings、LoRA、Prometheus 都不在主链路上；去掉它们服务照样跑，加上它们是为了不让业务绕路。

**组件 1：部署拓扑（进程怎么放）**
- 本机：`vllm-metal` 提供单进程服务，8000 端口；
- 云上：`vllm/vllm-openai` 容器暴露 8000，容器内 API Server + EngineCore，GPU worker 由 `--tensor-parallel-size` 决定；
- 附加件：HF cache 挂载（避免重下 16GB）、LoRA 适配器目录、独立 embeddings 实例（`--task embed`，另一个端口）。

HF cache 这一项最容易被忽略：8B 模型约 16GB，不挂 volume，每重启一次容器就重下一次——
**调试时你重启容器的次数，总是比计划里多**。

**组件 2：API 契约（客户端只见协议）**
本章只依赖 4 个入口：`/v1/chat/completions`、`/v1/completions`、`/v1/embeddings`、`/health`。
请求字段与响应字段见下面的契约表；实现细节（LoRA adapter、guided decoding、tool parser）都藏在服务端。

**组件 3：客户端降级（没有服务也要能跑）**
`VLLMClient` 启动时探测 `/health`：200 → 在线模式，走真实 HTTP 流式；
否则 → 回放模式，按 fixture 顺序返回录制响应，并在表里标注「回放」。
两种模式返回**同构 dict**，上层实验代码不区分。

降级不是退而求其次：它把「契约」和「实现」解耦了——契约由 fixture 钉死，实现（真服务）
随时可以换。于是同一批断言，在没有 GPU 的 Mac 上也能跑。

数据流：

```text
业务请求 → (在线) HTTP/SSE / (回放) fixture ──→ 统一 dict → 断言 / 表格 / 图
```

### 接口契约

| Endpoint | 关键请求字段 | 关键响应字段 | 用途 |
| --- | --- | --- | --- |
| `POST /v1/chat/completions` | `model` `messages` `max_tokens` `temperature` `stream` `tools` `tool_choice` `response_format` | `choices[].message`、`finish_reason`、`usage` | 对话 / 工具调用 / 结构化输出 |
| `POST /v1/completions` | `model` `prompt` `max_tokens` `n` `stream` | `choices[].text`、`usage` | 纯文本补全 |
| `POST /v1/embeddings` | `model` `input` `encoding_format` | `data[].embedding`、`usage` | 向量化（需 `--task embed`） |
| `GET /health` | — | 200 = 就绪 | 客户端探测 / 负载均衡 |

客户端方法（在线 / 回放两种实现）：

| 方法 | 在线行为 | 回放行为 |
| --- | --- | --- |
| `chat(payload, idx)` | POST 流式，记录首块时间与 token 数 | 返回 `records[idx]` 的录制响应 |
| `embed(record)` | POST `/v1/embeddings` | 返回录制响应 |
| `mode()` | `"在线"` | `"回放"` |

### 验收标准

| 步骤 | 验收 |
| --- | --- |
| Step 1 启动命令 | 三套命令 + 生产参数打印完整，`--enable-lora` 等关键开关存在 |
| Step 2 fixture / 环境 | 三个 JSON 可读；`httpx` 缺失时客户端仍可回放 |
| Step 3 客户端 | 无服务时 `online=False`，`chat/embed` 返回 200 记录且不抛异常 |
| Step 4 高级能力 | JSON Schema 可解析、`tool_calls` 形状正确、embedding 维度正确 |
| 实验 A/B/C | 10 条 smoke = 9 + 1；显存表可复算；成本表可复算 |

## 10.3 从零实现：分四步

| Step | 做什么 | 代码 / 命令 |
| --- | --- | --- |
| 1 | 起服务：Mac / 云 / compose / 生产参数 | 命令生成 + `PROD_FLAGS` |
| 2 | 环境与回放数据 | imports + fixture 校验 |
| 3 | 可降级客户端 | `_post_stream` / `VLLMClient` |
| 4 | 验证高级能力 | JSON Schema / `tool_calls` / embeddings |

### Step 1 · 起服务命令

目标：给出三条可复制的启动路径，以及一条「能力全开」的生产参数。

做法：三种方式任选其一，**不要同时跑**（8000 端口会冲突）；首次启动要下载模型（8B 约 16GB），
看到 `Application startup complete` 后用 `curl $BASE/health` 验证。

In [ ]:
MODEL = "Qwen/Qwen3-8B"
SERVE = {
    "Mac（vllm-metal，单机体验）": [
        "brew tap vllm-project/vllm-metal",
        "brew install vllm-metal",
        f"vllm serve {MODEL} --port 8000 --max-model-len 4096",
    ],
    "云上（官方镜像 + compose）": [
        "cd cloud",
        "docker compose up -d",
        "curl -s http://127.0.0.1:8000/health",
    ],
    "云上（裸 docker run，临时调试）": [
        "docker run --rm --gpus all -p 8000:8000 \\",
        "  -v ~/.cache/huggingface:/root/.cache/huggingface \\",
        f"  vllm/vllm-openai:v0.28.0 --model {MODEL}",
    ],
}
for mode, cmds in SERVE.items():
    print(f"[{mode}]")
    for c in cmds:
        print("  " + c)

生产上不会只给一个 `--model`。下面是一条「能力全开」的命令，每个参数都在本章后面有对应小节：

- `--enable-lora`：开多适配器；`--max-loras` / `--max-lora-rank` 是容量上限；
- `--enable-auto-tool-choice --tool-call-parser hermes`：让模型按工具 schema 返回 `tool_calls`；
- `--structured-outputs-config.backend xgrammar`：约束解码；请求侧用 `response_format` 触发；
- `--gpu-memory-utilization` / `--max-num-seqs` / `--max-num-batched-tokens`：第 11 章重点调的三个旋钮。

In [ ]:
PROD_FLAGS = f"""vllm serve {MODEL} \\
  --host 0.0.0.0 --port 8000 \\
  --max-model-len 8192 \\
  --gpu-memory-utilization 0.90 \\
  --max-num-seqs 256 --max-num-batched-tokens 8192 \\
  --enable-prefix-caching --enable-chunked-prefill \\
  --enable-lora --max-loras 4 --max-lora-rank 64 \\
  --enable-auto-tool-choice --tool-call-parser hermes \\
  --structured-outputs-config.backend xgrammar \\
  --api-key YOUR_KEY"""
print(PROD_FLAGS)
assert "--enable-lora" in PROD_FLAGS and "--enable-auto-tool-choice" in PROD_FLAGS

**Step 1 验收**：三套命令都打印完整；生产参数里包含 `--enable-lora`、`--enable-auto-tool-choice`
等关键开关（`assert` 通过）。命令本身不在 notebook 里执行，上云时照抄即可。

### Step 2 · 环境与回放数据

目标：让 notebook 在没有任何服务时也能跑通。

做法：依赖只有标准库 + numpy + pandas；`httpx` 可选（没装就强制回放）。
数据来自 `cloud/fixtures/` 的三个 JSON：chat 响应、embeddings、benchmark 结果。

In [ ]:
import json
import time
from pathlib import Path

import numpy as np
import pandas as pd

try:
    import httpx
except ImportError:          # 没装 httpx 也能回放；装了才可能走在线
    httpx = None

from aiinfra.bench import human_bytes, kv_cache_bytes
from aiinfra.plotting import save, setup_style

setup_style()
ROOT = Path.cwd().parent
FIXTURES = ROOT / "cloud" / "fixtures"
print("fixtures:", sorted(p.name for p in FIXTURES.glob("*.json")))
assert (FIXTURES / "chat_completions.json").exists()

**Step 2 验收**：三个 fixture 都能被 `json` 读入；`httpx` 缺失时被置为 `None`，
客户端会自动强制回放（下一步实测）。

### Step 3 · 可降级客户端

目标：把「在线 HTTP 流式」与「fixture 回放」封装成同一个接口。

做法：`_post_stream` 记录首 token 时间与 token 数（一个 SSE data 块 ≈ 一个 token）；
`VLLMClient` 启动时探测 `/health`，决定走哪条路径。回放模式的提示会打印启动命令，
起好服务后重跑本 cell 即可自动切到在线模式，客户端代码不用改。

In [ ]:
def _post_stream(base_url: str, path: str, payload: dict) -> dict:
    """流式 POST：记录首 token 时间与 token 数（一个 SSE data 块 ≈ 一个 token）。"""
    t0 = time.perf_counter()
    with httpx.Client(timeout=60) as client:
        with client.stream("POST", base_url + path, json=payload) as r:
            if r.status_code != 200:
                return {"status_code": r.status_code, "ttft_ms": None,
                        "body": r.read().decode(), "n_tokens": 0, "replay": False}
            ttft, n_tokens = None, 0
            for line in r.iter_lines():
                if not line.startswith("data: ") or "[DONE]" in line:
                    continue
                if ttft is None:
                    ttft = (time.perf_counter() - t0) * 1000
                json.loads(line[6:])
                n_tokens += 1
    return {"status_code": 200, "ttft_ms": ttft, "body": None,
            "n_tokens": n_tokens, "replay": False}

In [ ]:
class VLLMClient:
    """OpenAI 兼容客户端：有服务走真实 HTTP，没有服务回放 cloud/fixtures。"""

    def __init__(self, base_url: str = "http://127.0.0.1:8000", fixtures: Path = FIXTURES):
        self.base_url = base_url.rstrip("/")
        self.chat_fx = json.loads((fixtures / "chat_completions.json").read_text(encoding="utf-8"))
        self.embed_fx = json.loads((fixtures / "embeddings.json").read_text(encoding="utf-8"))
        self.online = False
        if httpx is not None:
            try:
                self.online = httpx.get(self.base_url + "/health", timeout=0.4).status_code == 200
            except Exception:
                self.online = False

    def mode(self) -> str:
        return "在线" if self.online else "回放"

    def chat(self, payload: dict, idx: int) -> dict:
        if not self.online:
            rec = self.chat_fx["records"][idx]
            usage = rec["response"].get("usage", {})
            return {"status_code": rec["status_code"], "ttft_ms": rec["ttft_ms"],
                    "body": rec["response"], "n_tokens": usage.get("completion_tokens", 0),
                    "replay": True}
        return _post_stream(self.base_url, "/v1/chat/completions", payload | {"stream": True})

    def embed(self, record: dict) -> dict:
        if not self.online:
            return {"status_code": record["status_code"], "ttft_ms": record["latency_ms"],
                    "body": record["response"], "n_tokens": 0, "replay": True}
        r = httpx.post(self.base_url + "/v1/embeddings",
                       json={"model": record["response"]["model"], "input": record["input"]},
                       timeout=30)
        return {"status_code": r.status_code, "ttft_ms": None, "body": r.json(),
                "n_tokens": 0, "replay": False}

In [ ]:
client = VLLMClient()
if not client.online:
    print("WARNING 当前为回放模式：未检测到 http://127.0.0.1:8000/health")
    print("   起服务（Mac）: brew tap vllm-project/vllm-metal && brew install vllm-metal")
    print("                  vllm serve Qwen/Qwen3-8B --port 8000")
    print("   起服务（云） : cd cloud && docker compose up -d")
    print("   起好后重跑本 cell，会自动切换为在线模式（客户端代码不用改）。")
print("客户端模式 =", client.mode())
print("fixture 录制说明：", client.chat_fx["provenance"]["note"][:48], "...")

**Step 3 验收**：无服务时 `client.online is False`，`mode()` 返回「回放」，
并打印 Mac / 云上的启动命令；有服务时同一份代码切到在线模式。

### Step 4 · 高级能力验收

目标：用 fixture 里的真实响应验证三种能力在 JSON 里的形状。

做法：结构化输出（`response_format`）→ 合法 JSON；工具调用 → `tool_calls` +
`finish_reason="tool_calls"`；embeddings → `data[].embedding` 维度正确。
在线模式下这三条断言同样成立，所以回放不是「假测试」，而是同一契约的录制证据。

In [ ]:
json_rec = next(r for r in client.chat_fx["records"] if r["case"] == "json-schema")
parsed = json.loads(json_rec["response"]["choices"][0]["message"]["content"])
assert set(parsed) == {"model", "weights_gb", "kv_budget_gb"}
print("structured output（response_format=json_schema）解析成功：", parsed)

tool_rec = next(r for r in client.chat_fx["records"] if r["case"] == "tool-call")
call = tool_rec["response"]["choices"][0]["message"]["tool_calls"][0]
assert call["function"]["name"] == "get_weather"
print("tool calling：", call["function"]["name"], call["function"]["arguments"],
      "| finish_reason =", tool_rec["response"]["choices"][0]["finish_reason"])

In [ ]:
emb = client.embed(client.embed_fx["records"][0])
vec = np.asarray(emb["body"]["data"][0]["embedding"], dtype=float)
assert vec.shape[0] == client.embed_fx["embedding_dim"]
pd.DataFrame([{"输入": client.embed_fx["records"][0]["input"], "维度": vec.shape[0],
               "前 4 维": np.round(vec[:4], 3).tolist(), "首包(ms)": emb["ttft_ms"],
               "usage": emb["body"]["usage"]}])

**Step 4 验收**：JSON Schema 能解析成 dict；`tool_calls[0].function.name == "get_weather"`
且 `finish_reason == "tool_calls"`；embedding 维度等于 fixture 声明的 16。

## 10.4 实测

三个实验，全部在**无服务回放模式**下完成；数据来源是 `cloud/fixtures/`
（示例录制：字段结构复刻 vLLM 0.28.x 真实响应，数值为示例量级，非本机实测）。

**先猜一下**：10 条 smoke 里唯一失败的那条，会以什么方式失败？
(a) 模型拒答，返回 200 但内容为空　(b) 400，而且发生在进入引擎之前　(c) 超时。

| 实验 | 问题 | 产出 |
| --- | --- | --- |
| A | 10 种典型请求能不能按契约返回？ | smoke 表（状态码 / 首包 / token 数） |
| B | Qwen3-8B / 27B 在不同精度和卡上能开多少 KV？ | 显存表 + 两张图 |
| C | 每百万 token 多少钱？ | 成本表 |

测量协议：回放模式直接读取 fixture 的录制字段；在线模式由客户端实测（流式首块 = TTFT）。
所有表都在 Mac CPU 上复算，公式与 fixture 一起构成可复现的账本。

### 实验 A · 10 条 smoke test

协议：8 种正常 chat 场景（基础对话、长上下文、JSON Schema、工具调用、LoRA、temperature=0、
max_tokens 截断、中英混合）+ 1 条按预期失败的请求（`max_tokens` 超上下文）+ 1 条 embeddings。
回放模式的「首 token / 首包时间」和「输出 tokens」来自录制字段；在线模式由客户端实测。

In [ ]:
rows = []
cases = [(rec["case"], "chat", i) for i, rec in enumerate(client.chat_fx["records"])]
cases.append((client.embed_fx["records"][0]["case"], "embed", 0))
for n, (case, kind, idx) in enumerate(cases, 1):
    if kind == "chat":
        rec = client.chat_fx["records"][idx]
        out = client.chat(rec["request"], idx)
    else:
        rec = client.embed_fx["records"][idx]
        out = client.embed(rec)
    status = out["status_code"]
    note = "OK" if status == 200 else str(out["body"])[:64]
    rows.append({"#": n, "场景": case, "状态码": status,
                 "首 token/首包(ms)": None if out["ttft_ms"] is None else round(out["ttft_ms"], 1),
                 "输出 tokens": out["n_tokens"], "模式": client.mode(), "备注": note})
smoke_df = pd.DataFrame(rows)
smoke_df

In [ ]:
assert len(smoke_df) == 10, "smoke test 应有 10 条请求"
assert (smoke_df["状态码"] == 200).sum() == 9, "应有 9 条成功"
assert (smoke_df["状态码"] == 400).sum() == 1, "应有 1 条按预期返回 400"
assert smoke_df["首 token/首包(ms)"].notna().sum() == 9, "成功请求都应有首包时间"
ok = smoke_df[smoke_df["状态码"] == 200]
print(f"成功 {len(ok)}/10；首包中位数 {ok['首 token/首包(ms)'].median():.1f} ms；"
      f"输出 token 合计 {int(ok['输出 tokens'].sum())}")

**读数**：回收猜测——答案是 (b)。9 条成功、1 条 400，首包中位数约 172 ms，输出 token 合计 488。
错误请求返回 400 且带 `BadRequestError`。选 (a) 的人默认了「是模型在拒绝」，但模型根本没机会
看到这条请求：校验由 API Server 在引擎之前完成，坏请求连 GPU 都没碰到。

这正是 API Server 必须独立于引擎的原因——**校验失败的成本应该接近零**，
而让一条坏请求走进引擎，代价是实打实的一步 prefill。

### 实验 B · 显存账本：权重 × 精度 × `gpu_memory_utilization`

**先猜一下**：8B 模型 fp16 放到 24GB 卡上，权重占掉 16.4GB，剩下的显存够几个 2048 上下文的并发？
(a) 30 个上下　(b) 10 个上下　(c) 一个都放不下。

协议：按三条公式估算，再用 `kv_cache_bytes` 复算，不依赖任何真实服务。

```text
权重显存 = 参数量 × 每参数字节（fp16=2, int8=1, int4=0.5）
KV 显存 = 2 × layers × kv_heads × head_dim × 2 字节 × token 数
可用 KV = 卡容量 × gpu_memory_utilization − 权重 − 框架/激活预留(约 2GB)
```

**关键陷阱**：量化只压缩权重，**不压缩 KV cache**（KV 默认仍是 fp16/bf16）。
所以 int4 的 8B 在 24GB 卡上能跑，KV 空间主要来自「省下来的权重」。

把账换算成人话：8B 在 fp16 下权重 16.4 GB，80GB 卡按 util=0.90 有 72 GB 可用，
减掉 2 GB 预留，留给 KV 的是 53.6 GB；换成 24GB 卡（可用 21.6 GB），只剩 3.2 GB。
**同一份权重，换一张卡，草稿纸从 53.6 GB 缩到 3.2 GB**——这就是「能跑」和「能开多少并发」
之间那道鸿沟。

In [ ]:
MODELS = {
    "Qwen3-8B": {"params": 8.2e9, "n_layer": 36, "n_kv_head": 8, "head_dim": 128},
    "假设 27B（64 层）": {"params": 27.0e9, "n_layer": 64, "n_kv_head": 8, "head_dim": 128},
}
WEIGHT_BYTES = {"fp16": 2, "int8": 1, "int4": 0.5}
UTIL, OVERHEAD_GB = 0.90, 2.0

rows = []
for name, m in MODELS.items():
    kv_per_tok = kv_cache_bytes(m["n_layer"], m["n_kv_head"], m["head_dim"], 1, 1, 2)
    for dt, wb in WEIGHT_BYTES.items():
        weights = m["params"] * wb / 1e9
        kv_80 = 80 * UTIL - weights - OVERHEAD_GB
        rows.append({"模型": name, "权重精度": dt, "权重(GB)": round(weights, 1),
                     "KV/token(KB)": round(kv_per_tok / 1024, 1),
                     "24GB 能放下": "是" if 24 * UTIL - weights - OVERHEAD_GB > 1 else "否",
                     "80GB KV 预算(GB)": round(kv_80, 1),
                     "80GB 可容纳 token": int(max(kv_80, 0) * 1e9 // kv_per_tok),
                     "80GB@2048ctx 并发": int(max(kv_80, 0) * 1e9 // kv_per_tok // 2048)})
df_mem = pd.DataFrame(rows)
df_mem

In [ ]:
rows = []
for name, m in MODELS.items():
    kv_per_tok = kv_cache_bytes(m["n_layer"], m["n_kv_head"], m["head_dim"], 1, 1, 2)
    weights = m["params"] * 2 / 1e9
    for util in [0.80, 0.85, 0.90, 0.95]:
        kv_gb = 80 * util - weights - OVERHEAD_GB
        rows.append({"模型": name, "gpu_memory_utilization": util,
                     "权重(GB)": round(weights, 1), "KV 预算(GB)": round(kv_gb, 1),
                     "可容纳 token": int(max(kv_gb, 0) * 1e9 // kv_per_tok),
                     "2048 ctx 并发": int(max(kv_gb, 0) * 1e9 // kv_per_tok // 2048)})
df_util = pd.DataFrame(rows)
df_util

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots()
labels = (df_mem["模型"] + " " + df_mem["权重精度"]).tolist()
w_gb = df_mem["权重(GB)"].to_numpy()
kv_gb = df_mem["80GB KV 预算(GB)"].to_numpy()
ax.bar(labels, w_gb, color="#2563EB", label="weights")
ax.bar(labels, kv_gb, bottom=w_gb, color="#7C3AED", label="KV budget")
ax.bar(labels, [OVERHEAD_GB] * len(df_mem), bottom=w_gb + kv_gb,
       color="#9CA3AF", label="framework / activation")
ax.axhline(80, color="#EF4444", ls="--", lw=1.5, label="80GB card capacity")
ax.set_ylabel("GPU memory (GB)")
ax.set_title("80GB / util=0.90: weight precision decides how much KV fits")
ax.tick_params(axis="x", rotation=12, labelsize=9)
ax.legend(ncol=4, fontsize=8.5, loc="upper center", bbox_to_anchor=(0.5, -0.14), frameon=False)
save(fig, "ch10_memory_budget")

In [ ]:
fig, ax = plt.subplots()
utils = np.linspace(0.75, 0.95, 9)
for name, m in MODELS.items():
    kv_per_tok = kv_cache_bytes(m["n_layer"], m["n_kv_head"], m["head_dim"], 1, 1, 2)
    weights = m["params"] * 2 / 1e9
    tokens = np.maximum(80 * utils - weights - OVERHEAD_GB, 0) * 1e9 / kv_per_tok
    ax.plot(utils, tokens / 1000, "o-", label=name)
ax.set_xlabel("gpu_memory_utilization")
ax.set_ylabel("KV token capacity (thousands)")
ax.set_title("Higher gpu_memory_utilization: more KV capacity, higher OOM risk")
ax.legend()
save(fig, "ch10_kv_vs_util")

**读数**：

- 8B 在 fp16 下：24GB 卡只剩约 3.2GB 给 KV（约 2.2 万 token、10 个 2048 上下文并发）；
  80GB 卡能到 36 万 token、177 个并发。这就是「同样模型换卡能多开 17 倍并发」的来源。
- 27B 在 fp16 下 54GB 权重，24GB/48GB 直接放不下；int4 后 13.5GB，才能在 24GB 卡上跑，
  但 KV 只剩约 6GB（11 个并发）。
- `gpu_memory_utilization` 从 0.80 调到 0.95，8B 的 KV 容量涨约 25%；再往上就会被 CUDA graph、
  激活和碎片吃掉——**0.90 是常用折中，0.95 以上只在确认无峰值时用**。

回收刚才那个猜测：答案是 (b)。3.2 GB KV 摊到每个 2048 上下文上，能同时放下的大约就是 10 个；
你想同时服务 10 个用户，他们的上下文就不能超过 2048，超出的部分要么排队、要么抢占。
选 (a) 的人算的是「24GB − 权重 16.4GB = 7.6GB」，忘了 util=0.90 只放出 21.6GB，
也忘了 2GB 的框架预留；两个折扣一打，7.6GB 变成 3.2GB，并发从三十个上下掉到十个上下。

> **⚠️ 坑：以为量化能省 KV。**
> 8B 从 fp16 换到 int4，权重从 16.4 GB 掉到 4.1 GB，省下 12 GB；但 KV cache 的 dtype
> 由 `--kv-cache-dtype` 决定，不跟权重精度走，默认还是 fp16。所以 int4 救的是「放得下模型」，
> 不是「能开更多并发」——省下的显存变成草稿纸，不会自动变成并发能力。

### 实验 C · 成本估算：每百万输出 token 多少钱

协议：用第 11 章 fixture 的回放吞吐做量级估算（8B 在 4090/A100/H100 上分别为 420/812/1350 tok/s；
27B 按参数量线性外推）。**价格是示例价，下单前以平台页面为准。**

In [ ]:
CARD_PRICE = {"4090 24GB": 0.69, "A100 80GB": 1.99, "H100 80GB": 2.99}   # USD/小时（示例）
REF_8B = {"4090 24GB": 420, "A100 80GB": 812, "H100 80GB": 1350}        # tok/s（回放量级）
rows = []
for card, price in CARD_PRICE.items():
    for model, toks in [("Qwen3-8B", REF_8B[card]), ("Qwen3-27B", REF_8B[card] * 8.2 / 27)]:
        rows.append({"卡型": card, "模型": model, "小时价(USD)": price,
                     "参考吞吐(tok/s)": round(toks),
                     "每百万输出 token(USD)": round(price / (toks * 3600) * 1e6, 2)})
df_cost = pd.DataFrame(rows)
df_cost

**读数**：**每 token 成本 ≈ 小时价 ÷ (吞吐 × 3600)**。

4090 的小时价只有 A100 的三分之一左右（0.69 vs 1.99），吞吐是它的一半左右（420 vs 812 tok/s）——
除下来，4090 每输出 token 反而更便宜。所以「贵不贵」不看卡的价格，看**你的并发能不能把卡喂饱**：
喂不饱的 H100，每 token 成本会比喂饱的 4090 还高。

这也接上了下一章：吞吐不是一张卡的固有属性，它随并发、KV、参数变化——第 11 章教你怎么把它压出来。

## 10.5 对照工业实现（vLLM 0.28.x）

### 映射表：我们的客户端 / 服务端 vs vLLM 实现

| 我们的实现 | vLLM 对应物 | 真实文件（0.28.x） |
| --- | --- | --- |
| `_post_stream` 的 SSE 解析 | API Server 流式返回 | `vllm/entrypoints/openai/api_server.py` + `serving_chat.py` |
| `VLLMClient.chat` | `POST /v1/chat/completions` | 同目录 `serving_chat.py` |
| `VLLMClient.embed` | `POST /v1/embeddings` | `serving_embedding.py`（需 `--task embed`，待核对） |
| `/health` 探测 | 健康检查端点 | `api_server.py` 的 `/health` |
| fixture 回放 | `vllm bench serve --save-result` 的产物 | `vllm/benchmarks/serve.py`（路径待核对） |

### 高级能力的开关与请求形状

| 能力 | 服务端参数 | 请求侧写法 | 响应侧特征 |
| --- | --- | --- | --- |
| LoRA 多适配器 | `--enable-lora --max-loras 4 --max-lora-rank 64 --lora-modules name=path` | `model` 填适配器名 | 与基座同结构，`model` 字段回显适配器名 |
| 动态加载 LoRA | 无需重启 | `POST /v1/load_lora_adapter`（vLLM 扩展） | 返回成功；`/v1/models` 可查 |
| 结构化输出 | `--structured-outputs-config.backend xgrammar`（旧版 `--guided-decoding-backend`，待核对） | `response_format={"type":"json_schema","json_schema":{...}}` | `message.content` 是合法 JSON |
| 引导解码（legacy） | 同上 | 请求里带 `guided_json` / `guided_regex` 等扩展字段 | 同上 |
| 工具调用 | `--enable-auto-tool-choice --tool-call-parser hermes` | `tools=[...]`、`tool_choice="auto"` | `choices[].message.tool_calls[]`，`finish_reason="tool_calls"` |

注意：`--tool-call-parser` 必须与模型训练时的工具格式匹配（Qwen 常用 `hermes`，Llama 常用 `llama3_json`）。
选错不会报错，只会退化成普通文本。

责任划分值得单独说清楚：**服务端负责把「约束」编译成 token 级许可**（xgrammar 在每一步
decode 时屏蔽非法 token），**客户端只负责描述 schema 和工具签名**。所以 schema 写错，报错在
客户端；而服务端开关没开，表现是悄悄退化，不报错。

> **⚠️ 坑：能力开关写成旧参数名，服务能起来，能力却静默关闭。**
> `--guided-decoding-backend` 和 `--structured-outputs-config.backend` 是两个版本的写法，
> `--tool-call-parser` 选错也不报错——现象是「接口通了、模型也回了，就是没有 `tool_calls`」。
> 这类问题排查方向不在模型，在启动参数表。

### 差距在哪：兼容层不是「加个路由」

一个能上生产的 OpenAI 兼容层还要处理：流式背压与客户端断连、`usage` 的精确统计（含缓存命中 token）、
错误码映射（400/422/429/503）、多模态输入、tokenizer 差异、请求取消（`abort`）、API Key 鉴权与限流。
这些在 vLLM 里散落在 `vllm/entrypoints/openai/` 与 `vllm/v1/engine/` 中，也是
「自己写个 FastAPI 包一层」和工业实现的主要差距。

## 10.6 练习与小结

**练习**（⭐ 改参数 / ⭐⭐ 改实现 / ⭐⭐⭐ 开放设计）

1. ⭐ 把 `--max-model-len` 从 8192 改成 4096，重跑 10.4 实验 B 的 KV 计算。并发数为什么不是翻倍？提示：`--max-model-len` 影响的是单请求上限，不是块总数。
2. ⭐ 用 `--gpu-memory-utilization 0.95` 起服务，再压 32 并发；观察是否 OOM 或吞吐下降。解释 0.95 为什么比 0.90 危险。
3. ⭐⭐ 给 `VLLMClient` 加 `completion()`（`/v1/completions`）与 `list_models()`，用 fixture 回放验证；再在真服务上各打一条。
4. ⭐⭐ 写一个 `load_lora_adapter(client, name, path)`：启动时 `--enable-lora`，运行中动态加载，然后对比「基座 / 适配器」对同一 prompt 的输出与 TTFT。
5. ⭐⭐⭐ 设计一份「同一服务同时提供 chat + embeddings + LoRA」的部署方案：几个进程、几个端口、GPU 怎么分、K8s 怎么探活、如何限流。（评分要点：资源隔离、端口与探活、失败降级、成本、监控 5 条）

**你现在应该能回答：**

- Mac 与云上的起服务命令差在哪？为什么客户端代码可以不变？
- 显存账本的三个公式是什么？为什么 int4 不省 KV cache？
- 什么情况下该用 `response_format`，什么情况下该用 `tool_calls`？两者能一起用吗？

**自检 3 问**（不看笔记能答出来才算过）

1. 在线模式和回放模式的**断言为什么可以完全相同**？如果 fixture 本身过时了，哪类问题会漏检？
2. LoRA、结构化输出、工具调用三件事，哪些是服务端责任、哪些是客户端责任？
3. 显存账本的三个公式分别是什么？为什么 int4 不省 KV cache？

> **一句话记住**：兼容层不是「多写一个路由」，而是把引擎的内部对象翻译成一份**别人可以依赖的契约**。

**真实工作场景题**

业务方反馈：「接上你们的服务后，`stream=True` 时前端偶尔会卡住，非流式完全正常。」
手上有本章的 10 条 smoke test 和 SSE 解析代码。写出你的排查顺序（先看哪一侧、
怎么区分服务端流式与客户端拼接的问题），以及如何用 fixture 回放复现。
**小结与下一章**

到这里服务已经能接真实流量了。但「能跑」不等于「跑得好」：并发加到哪里开始亏、
TTFT 和 TPOT 谁先崩、瓶颈在算力还是 KV——下一章用压测和决策树回答。

**延伸阅读**

- vLLM OpenAI 兼容服务器文档（`docs/serving/openai_compatible_server.md`，版本随 0.28.x）；
- `vllm/entrypoints/openai/serving_chat.py`：`response_format` 与 tool parser 的分支；
- `cloud/README.md`：AutoDL / RunPod / Lambda 的开机、起服务、压测、关机全流程；
- 第 11 章：压测、指标与调参决策树。